# CloudPulse Phase 7 Statistical Evaluation (`T7.4` & `T7.5`)

Evaluates all committed runs in `analysis/results.csv` using non-parametric statistics:
- Median and IQR per `(scaler, ablation, profile, warmup_seconds)` cell with explicit `n` and `run_id` provenance
- Kruskal–Wallis H-test + tie-corrected Dunn post-hoc (`p_adj`, Holm–Bonferroni), Cliff's delta effect size, and bootstrap 95% median-difference confidence intervals
- Startup-latency sweep break-even curve (`WARMUP_SECONDS` in `{0, 5, 15, 30, 60}`, finding both where CloudPulse wins and below what warmup reactive HPA wins)
- Primary cost metric: `pod_seconds` (D14)

In [ ]:
from pathlib import Path
import pandas as pd

from analysis.stats import (
    analyze_warmup_breakeven,
    kruskal_dunn_posthoc,
    render_benchmark_markdown,
    summarize_cells,
)

RESULTS_CSV = Path("analysis/results.csv")
df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
print(f"Loaded {len(df)} runs from {RESULTS_CSV}")
display(df.head())

In [ ]:
summary_df = summarize_cells(df)
display(summary_df)

In [ ]:
if not df.empty:
    for (profile, warmup_s), slice_df in df[df["ablation"].isna()].groupby(["profile", "warmup_seconds"], sort=True):
        for metric in ("p99_latency_ms", "sla_violation_pct", "pod_seconds"):
            res = kruskal_dunn_posthoc(slice_df, metric=metric)
            print(f"=== {profile} warmup={warmup_s}s | {metric}: H={res['h_stat']}, p={res['p_value']:.4g}, eps2={res['epsilon_sq']}, n={res['n_total']} ===")
            if res["pairwise"]:
                display(pd.DataFrame(res["pairwise"]))

In [ ]:
if not df.empty:
    for profile in sorted(df["profile"].dropna().unique()):
        sweep = analyze_warmup_breakeven(df, profile=str(profile))
        print(sweep["finding"])
        if sweep["curve"]:
            display(pd.DataFrame(sweep["curve"]))
    print("\n" + render_benchmark_markdown(df))